# Pandas — What You Actually Need

**One job:** Load a dataset → inspect it → clean it → hand it to a model.

That is all pandas does for you as an AI engineer. This notebook covers exactly that — nothing else.

In [1]:
import pandas as pd
import numpy as np

---
## 1. Load & Inspect

Every real ML dataset starts as a CSV (or similar file). `read_csv` is the first line of every data pipeline you'll ever write.

In [2]:
df = pd.read_csv('students.csv')
df

,name,age,cgpa,internships,primary_skill,projects,placed
0,Alice,21,8.5,2,Python,3,1
1,Bob,22,7.2,1,Java,1,0
2,Charlie,21,9.1,3,Python,5,1
3,Diana,23,6.8,0,C++,1,0
4,Eve,22,8.9,2,Python,4,1
5,Frank,21,7.5,1,Java,2,1
6,Grace,23,9.5,4,Python,6,1
7,Henry,22,6.2,0,C++,0,0
8,Iris,21,8.1,2,Python,3,1
9,Jack,22,7.8,1,Java,2,0


In [3]:
# Always run these 4 on any new dataset — in this order

print(df.shape)        # (rows, columns) — how big is this?
print(df.columns.tolist())  # what are the features?

(20, 7)
['name', 'age', 'cgpa', 'internships', 'primary_skill', 'projects', 'placed']


In [4]:
df.head()   # first 5 rows — did it load correctly?

,name,age,cgpa,internships,primary_skill,projects,placed
0,Alice,21,8.5,2,Python,3,1
1,Bob,22,7.2,1,Java,1,0
2,Charlie,21,9.1,3,Python,5,1
3,Diana,23,6.8,0,C++,1,0
4,Eve,22,8.9,2,Python,4,1


In [5]:
df.info()   # column types + how many non-null values — reveals missing data instantly

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           20 non-null     str    
 1   age            20 non-null     int64  
 2   cgpa           20 non-null     float64
 3   internships    20 non-null     int64  
 4   primary_skill  20 non-null     str    
 5   projects       20 non-null     int64  
 6   placed         20 non-null     int64  
dtypes: float64(1), int64(4), str(2)
memory usage: 1.2 KB


In [6]:
df.describe()   # mean, std, min, max for every numeric column — your data at a glance

,age,cgpa,internships,projects,placed
count,20.000000,20.000000,20.000000,20.000000,20.000000
mean,21.850000,7.830000,1.600000,2.700000,0.600000
std,0.812728,1.077082,1.313893,1.838191,0.502625
min,21.000000,5.800000,0.000000,0.000000,0.000000
25%,21.000000,6.975000,0.750000,1.000000,0.000000
50%,22.000000,7.900000,1.500000,2.500000,1.000000
75%,22.250000,8.750000,2.250000,4.000000,1.000000
max,23.000000,9.500000,4.000000,6.000000,1.000000


In [7]:
# Check the target column — how balanced are the classes?
# Class imbalance is the first thing you check before training any classifier
df['placed'].value_counts()

placed
1    12
0     8
Name: count, dtype: int64

---
## 2. Select What You Need

You'll spend 80% of your time doing this one thing: picking which columns are features (X) and which is the target (y).

In [8]:
# Single column → Series (1D)
cgpa = df['cgpa']

# Multiple columns → DataFrame (2D)
features = df[['cgpa', 'internships', 'projects']]

print(type(cgpa))     # Series
print(type(features)) # DataFrame

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


In [9]:
# THE most important pandas operation for ML — split features from target
X = df[['cgpa', 'internships', 'projects']]  # feature matrix
y = df['placed']                              # target vector

print('X:', X.shape)   # (20, 3)
print('y:', y.shape)   # (20,)

X: (20, 3)
y: (20,)


---
## 3. Filter Rows

Same idea as NumPy boolean masking — just with column names instead of indices.

In [10]:
# Single condition
high_cgpa = df[df['cgpa'] >= 9.0]
high_cgpa[['name', 'cgpa', 'placed']]

,name,cgpa,placed
2,Charlie,9.1,1
6,Grace,9.5,1
12,Maya,9.0,1
16,Quinn,9.2,1


In [11]:
# Compound condition — wrap each condition in ( )
# & = AND,  | = OR,  ~ = NOT
strong = df[(df['cgpa'] >= 8.5) & (df['internships'] >= 2)]
strong[['name', 'cgpa', 'internships', 'placed']]

,name,cgpa,internships,placed
0,Alice,8.5,2,1
2,Charlie,9.1,3,1
4,Eve,8.9,2,1
6,Grace,9.5,4,1
10,Kate,8.7,3,1
12,Maya,9.0,3,1
16,Quinn,9.2,4,1


---
## 4. Handle Missing Values

ML models crash on NaN. Always check and fix before doing anything else.

In [12]:
# Simulate missing data (very common in real datasets)
df_messy = df.copy()
df_messy.loc[[1, 5, 12], 'cgpa'] = np.nan
df_messy.loc[[3, 9], 'internships'] = np.nan

In [13]:
# Step 1 — count NaNs per column
df_messy.isna().sum()

name             0
age              0
cgpa             3
internships      2
primary_skill    0
projects         0
placed           0
dtype: int64

In [ ]:
df_clean = df_messy.copy()

# Option A — fill with column mean (standard for numeric features)
df_clean['cgpa'] = df_clean['cgpa'].fillna(df_clean['cgpa'].mean())
df_clean['internships'] = df_clean['internships'].fillna(df_clean['internships'].median())

# Option B — just drop rows with NaN (use when you have enough data)
# df_clean = df_messy.dropna()

print('NaNs left:', df_clean.isna().sum().sum())  # should be 0

---
## 5. Create New Columns (Feature Engineering)

Combine existing columns into a more useful feature. This is how you improve models without more data.

In [ ]:
df_work = df.copy()

# Arithmetic — combine two columns into a new feature
df_work['experience_score'] = df_work['internships'] * 10 + df_work['projects'] * 5

# apply() — run any function on each value in a column
df_work['cgpa_label'] = df_work['cgpa'].apply(lambda x: 'high' if x >= 8.5 else 'low')

df_work[['name', 'cgpa', 'cgpa_label', 'internships', 'projects', 'experience_score']].head(6)

---
## 6. GroupBy — Quick EDA Before Training

Before touching a model, you want to know: *do my features actually correlate with the target?* GroupBy answers this in one line.

In [ ]:
# Average feature values for placed vs not-placed
# If means are very different → the feature is useful for prediction
df.groupby('placed')[['cgpa', 'internships', 'projects']].mean().round(2)

In [ ]:
# Placement rate by skill — is Python actually better?
df.groupby('primary_skill')['placed'].mean().mul(100).round(1).rename('placement_%')

---
## 7. Hand Off to NumPy — The End Goal

scikit-learn, PyTorch, and TensorFlow all want NumPy arrays, not DataFrames. This is the last step of every pandas pipeline.

In [ ]:
# The complete pipeline — 7 lines from raw CSV to model-ready arrays

df = pd.read_csv('students.csv')                          # 1. load
df = df.dropna()                                          # 2. drop NaNs

X = df[['cgpa', 'internships', 'projects']].to_numpy()   # 3. features → numpy
y = df['placed'].to_numpy()                               # 4. target → numpy

X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))  # 5. normalize [0,1]

print('X shape:', X.shape, '| dtype:', X.dtype)
print('y shape:', y.shape)
print('\nFirst 3 rows of X (normalized):')
print(X[:3].round(3))

---
## Cheat Sheet

| What | Code |
|---|---|
| Load CSV | `df = pd.read_csv('file.csv')` |
| Size | `df.shape` |
| Column names | `df.columns.tolist()` |
| First rows | `df.head()` |
| Types + nulls | `df.info()` |
| Stats summary | `df.describe()` |
| Value counts | `df['col'].value_counts()` |
| Single column | `df['col']` |
| Multiple columns | `df[['c1', 'c2']]` |
| Filter rows | `df[df['col'] > val]` |
| AND / OR filter | `df[(cond1) & (cond2)]` |
| Count NaNs | `df.isna().sum()` |
| Fill NaN | `df['col'].fillna(df['col'].mean())` |
| Drop NaN rows | `df.dropna()` |
| New column | `df['new'] = df['a'] + df['b']` |
| Apply function | `df['col'].apply(lambda x: ...)` |
| Group stats | `df.groupby('col')['target'].mean()` |
| → NumPy | `df[['c1','c2']].to_numpy()` |
| Save CSV | `df.to_csv('out.csv', index=False)` |

In [ ]:
# practice here